<a href="https://colab.research.google.com/github/haydencase0/MachineLearning/blob/main/notebooks/Exploration_02_Polars.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Exploration 02

You're working as a data analyst at a cereal marketing company in New York.

In a strategy meeting, the marketing director tells you that in 2018, the US weight loss industry was worth over $72 Billion dollars, growing 4% compared to the previous year.

In contrast, sales of cold cereal fell 6% to $8.5 billion during the same time period.

Cereal executives have approached the marketing company asking how they can somehow tap into the weight loss market growth to boost the sales of their cereal brands.

Your assignment is to analyze a dataset of nutritional information for major US cereals, and calculate some metrics that can be used by the marketing team.

## Part 1: Import Polars and load the data

Remember to import Polars the conventional way. If you've forgotten how, you may want to review [Data Exploration 01](https://byui-cse.github.io/cse450-course/module-01/exploration-01.html).

The dataset for this exploration is stored at the following url:

[https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/cereal.csv](https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/cereal.csv)

There are lots of ways to load data into your workspace. The easiest way in this case is to [ask Polars to do it for you](https://docs.pola.rs/user-guide/getting-started/#reading-writing).

### Initial Data Analysis
Once you've loaded the data, it's a good idea to poke around a little bit to find out what you're dealing with.

Some questions you might ask include:

* What does the data look like?
* What kind of data is in each column?
* Do any of the columns have missing values?

In [18]:
# Part 1: Enter your code below to import Polars according to the
# conventional method. Then load the dataset into a Polars dataframe.

# Write any code needed to explore the data by seeing what the first few
# rows look like. Then display a technical summary of the data to determine
# the data types of each column, and which columns have missing data.


In [19]:
import polars as pl

data = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/cereal.csv")
data.head()

name,mfr,type,calories,protein,fat,sodium,fiber,carbo,sugars,potass,vitamins,shelf,weight,cups,rating
str,str,str,i64,i64,i64,i64,f64,f64,i64,i64,i64,i64,f64,f64,f64
"""100% Bran""","""N""","""C""",70,4,1,130,10.0,5.0,6,280,25,3,1.0,0.33,68.402973
"""100% Natural Bran""","""Q""","""C""",120,3,5,15,2.0,8.0,8,135,0,3,1.0,1.0,33.983679
"""All-Bran""","""K""","""C""",70,4,1,260,9.0,7.0,5,320,25,3,1.0,0.33,59.425505
"""All-Bran with Extra Fiber""","""K""","""C""",50,4,0,140,14.0,8.0,0,330,25,3,1.0,0.5,93.704912
"""Almond Delight""","""R""","""C""",110,2,2,200,1.0,14.0,8,-1,25,3,1.0,0.75,34.384843


## Part 2: Calculate Summary Statistics

The marketing team has determined that when choosing a cereal, consumers are most interested in `calories`, `sugar`, `fiber`, `fat`, and `protein`.

First, let's calcuate some summary statistics for these categories across the entire dataset. We're particularly intrested in the mean, median, standard deviation, min, and max values.

There are [multiple ways to accomplish this](https://docs.pola.rs/user-guide/concepts/data-types-and-structures/#describe).

In [20]:
# Part 2: Enter your code below to calculate summary statistics for the
# calories, sugar, fiber, fat, and protein features.

data.select(['calories', 'sugars', 'fiber', 'fat', 'protein']).describe()

statistic,calories,sugars,fiber,fat,protein
str,f64,f64,f64,f64,f64
"""count""",77.0,77.0,77.0,77.0,77.0
"""null_count""",0.0,0.0,0.0,0.0,0.0
"""mean""",106.883117,6.922078,2.151948,1.012987,2.545455
"""std""",19.484119,4.444885,2.383364,1.006473,1.09479
"""min""",50.0,-1.0,0.0,0.0,1.0
"""25%""",100.0,3.0,1.0,0.0,2.0
"""50%""",110.0,7.0,2.0,1.0,3.0
"""75%""",110.0,11.0,3.0,2.0,3.0
"""max""",160.0,15.0,14.0,5.0,6.0


## Part 3: Transform Data

To make analysis easier, you want to convert the manufacturer codes used in the dataset to the manufacturer names.

First, display the count of each manufacturer code value used in the dataset (found in the `mfr` column).

Then, [create a new column with the appropriate manufacturer name for each entry](https://docs.pola.rs/user-guide/concepts/expressions-and-contexts/#contexts), using this mapping:

    A = American Home Food Products
    G = General Mills
    K = Kelloggs
    N = Nabisco
    P = Post
    Q = Quaker Oats
    R = Ralston Purina

> Note: Some options include [`replace`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.replace.html#polars.Expr.replace), [`when`/`then`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.when.html#polars.when),  or even [`map_elements`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.map_elements.html#polars.Expr.map_elements).

In [21]:
# Display the count of values for the manufacturer code ("mfr" column), then
# create a new column containing the appropriate manufacturer names.
manufacturer = data.with_columns(
    pl.when(pl.col("mfr") == "A").then(pl.lit("American Home Food Products"))
    .when(pl.col("mfr") == "G").then(pl.lit("General Mills"))
    .when(pl.col("mfr") == "K").then(pl.lit("Kelloggs"))
    .when(pl.col("mfr") == "N").then(pl.lit("Nabisco"))
    .when(pl.col("mfr") == "P").then(pl.lit("Post"))
    .when(pl.col("mfr") == "Q").then(pl.lit("Quaker Oats"))
    .when(pl.col("mfr") == "R").then(pl.lit("Ralston Purina")).alias("mfr_name")
)
manufacturer.group_by(pl.col('mfr_name')).len()
manufacturer.head()

name,mfr,type,calories,protein,fat,sodium,fiber,carbo,sugars,potass,vitamins,shelf,weight,cups,rating,mfr_name
str,str,str,i64,i64,i64,i64,f64,f64,i64,i64,i64,i64,f64,f64,f64,str
"""100% Bran""","""N""","""C""",70,4,1,130,10.0,5.0,6,280,25,3,1.0,0.33,68.402973,"""Nabisco"""
"""100% Natural Bran""","""Q""","""C""",120,3,5,15,2.0,8.0,8,135,0,3,1.0,1.0,33.983679,"""Quaker Oats"""
"""All-Bran""","""K""","""C""",70,4,1,260,9.0,7.0,5,320,25,3,1.0,0.33,59.425505,"""Kelloggs"""
"""All-Bran with Extra Fiber""","""K""","""C""",50,4,0,140,14.0,8.0,0,330,25,3,1.0,0.5,93.704912,"""Kelloggs"""
"""Almond Delight""","""R""","""C""",110,2,2,200,1.0,14.0,8,-1,25,3,1.0,0.75,34.384843,"""Ralston Purina"""


## Part 4: Visualization

Let's do some more data exploration visually.

Import your visualization library of choice and set any needed configuration options.

In [22]:
# Import your visualization library

### Sugar Distribution

Marketing tells us that their surveys have revealed that sugar content is the number one concern of consumers when choosing cereal.

They would like to see the following visualizations:

*  A histogram plot of the sugar content in all cereals.

* A scatter plot showing the relationship between sugar and calories.

* A box plot showing the distribution of sugar content by manufacturer.

In [23]:
# Create the three visualzations requested by the the marketing team
# Install lets-plot if you'd like
%pip install lets-plot

In [24]:
from lets_plot import *
LetsPlot.setup_html()

In [25]:
hist = (
    ggplot(manufacturer, aes(x='sugars'))
    + labs(x='Sugar Content', y='Number of Cereal Brands')
    + ggtitle('Distribution of Sugar Content')
    + geom_histogram()
)
hist

In [26]:
scatter = (
    ggplot(manufacturer, aes(x='calories', y='sugars', size='fat'))
    + geom_point()
    + labs(
        title="Sugar vs Calories",
        x="Calories (kcal)",
        y="Sugar (g)",
        size="Fat (g)"
    )
    + scale_x_continuous(
        breaks=list(range(0, 161, 10)),
        limits=(0, 160)
        )
    + theme(
        plot_title=element_text(size=14, face='bold', hjust=0.5),
        axis_title=element_text(size=12, face='bold'),
        legend_title=element_text(size=12, face='bold'),
        axis_text_x=element_text(angle=0)
    )
)
scatter

In [27]:
box = (
    ggplot(manufacturer.sort('mfr_name'), aes(x='mfr_name', y='sugars', fill='mfr_name'))
    + geom_boxplot()
    + labs(
        title="Sugar Distribution by Manufacturer",
        y="Sugar (g)"
    )
    + scale_y_continuous(
        breaks=list(range(-2, 17, 2)),
        limits=(-2, 16)
    )
    + theme(
        plot_title=element_text(size=14, face='bold', hjust=0.5),
        axis_title_y=element_text(size=12, face='bold'),
        axis_title_x=element_blank(),
        axis_text_x=element_text(angle=75),
        legend_title=element_text(size=12, face='bold'),
        legend_position='none'
    )
    + ggsize(600, 600)
)
box

# Part 5: Dietary Calculations

The marketing team has been able to arrange a partnership between the popular Weight Watchers diet brand and Kelloggs cereal.

The Weight Watchers system assigns a point value to each food, and participants in the program are allotted a certain number of points each day.

One recent formula for calculating points is:

    (Calories * .0305) + (Fat * .275) + (Sugar * .12) - (Protein * .098)

With the final answer being rounded to the nearest integer.

First, [add a new column](https://docs.pola.rs/user-guide/concepts/expressions-and-contexts/#with_columns) with the Weight Watchers point calculation derived from the existing data.

**Be sure to round the number to the nearest int and store the data as an int, not as a float with 0 decimals.**

Then, [select a subset of the data](https://docs.pola.rs/user-guide/concepts/expressions-and-contexts/#filter) containing just cereals manufactured by Kelloggs.

Finally, calculate the same summary statistics for the points calculations as earlier (mean, median, standard deviation, min, and max).



In [28]:
manufacturer = manufacturer.with_columns(
    (
        pl.col('calories') * .0305
        + pl.col('fat') * .275
        + pl.col('sugars') * .12
        - pl.col('protein') * .098
    )
    .round()
    .cast(pl.Int64)
    .alias('ww_points')
)

kelloggs = manufacturer.filter(pl.col('mfr') == 'K')
kelloggs.select(pl.col('ww_points')).describe()

statistic,ww_points
str,f64
"""count""",23.0
"""null_count""",0.0
"""mean""",4.217391
"""std""",1.241572
"""min""",1.0
"""25%""",3.0
"""50%""",5.0
"""75%""",5.0
"""max""",7.0


## 🌟 Above and Beyond 🌟

The marketing team is pleased with what you've accomplished so far. They have a meeting with top cereal executives in the morning, and they'd like you to do as many of the following additional tasks as you have time for:

1. Weight Watchers used to have an older points system that used this formula: `(calories / 50) + (fat / 12) - (fiber / 5)`, but only the first 4 grams of fiber were included in the calculation. For comparison's sake, create an additional column with the calculation for the old points system.

2. Marketing really likes the boxplot of the sugar content for each cereal, they'd like similar plots for calories and fat, but using different color schemes for each chart.

In [29]:
manufacturer = manufacturer.with_columns(
    (
        pl.col('calories') / 50
        + pl.col('fat') / 12
        - pl.min_horizontal(pl.col('fiber'), 4) / 5
    )
    .round()
    .cast(pl.Int64)
    .alias('old_ww_points')
)
manufacturer

name,mfr,type,calories,protein,fat,sodium,fiber,carbo,sugars,potass,vitamins,shelf,weight,cups,rating,mfr_name,ww_points,old_ww_points
str,str,str,i64,i64,i64,i64,f64,f64,i64,i64,i64,i64,f64,f64,f64,str,i64,i64
"""100% Bran""","""N""","""C""",70,4,1,130,10.0,5.0,6,280,25,3,1.0,0.33,68.402973,"""Nabisco""",3,1
"""100% Natural Bran""","""Q""","""C""",120,3,5,15,2.0,8.0,8,135,0,3,1.0,1.0,33.983679,"""Quaker Oats""",6,2
"""All-Bran""","""K""","""C""",70,4,1,260,9.0,7.0,5,320,25,3,1.0,0.33,59.425505,"""Kelloggs""",3,1
"""All-Bran with Extra Fiber""","""K""","""C""",50,4,0,140,14.0,8.0,0,330,25,3,1.0,0.5,93.704912,"""Kelloggs""",1,0
"""Almond Delight""","""R""","""C""",110,2,2,200,1.0,14.0,8,-1,25,3,1.0,0.75,34.384843,"""Ralston Purina""",5,2
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Triples""","""G""","""C""",110,2,1,250,0.0,21.0,3,60,25,3,1.0,0.75,39.106174,"""General Mills""",4,2
"""Trix""","""G""","""C""",110,1,1,140,0.0,13.0,12,25,25,2,1.0,1.0,27.753301,"""General Mills""",5,2
"""Wheat Chex""","""R""","""C""",100,3,1,230,3.0,17.0,3,115,25,1,1.0,0.67,49.787445,"""Ralston Purina""",3,1


In [33]:
box_cal = (
    ggplot(manufacturer.sort('mfr_name'), aes(x='mfr_name', y='calories', fill='mfr_name'))
    + geom_boxplot()
    + labs(
        title="Calorie Distribution by Manufacturer",
        y="Calories (kcal)"
    )
    + scale_fill_brewer(palette='Set2')
    + theme(
        plot_title=element_text(size=14, face='bold', hjust=0.5),
        axis_title_y=element_text(size=12, face='bold'),
        axis_title_x=element_blank(),
        axis_text_x=element_text(angle=75),
        legend_title=element_text(size=12, face='bold'),
        legend_position='none'
    )
    + ggsize(600, 600)
)
box_cal

In [34]:
box_fat = (
    ggplot(manufacturer.sort('mfr_name'), aes(x='mfr_name', y='fat', fill='mfr_name'))
    + geom_boxplot()
    + labs(
        title="Fat Distribution by Manufacturer",
        y="Fat (g)"
    )
    + scale_fill_brewer(palette='Set3')
    + theme(
        plot_title=element_text(size=14, face='bold', hjust=0.5),
        axis_title_y=element_text(size=12, face='bold'),
        axis_title_x=element_blank(),
        axis_text_x=element_text(angle=75),
        legend_title=element_text(size=12, face='bold'),
        legend_position='none'
    )
    + ggsize(600, 600)
)
box_fat